# 第 1 周末练习 —— 职位描述技能提炼

## 练习目标

从招聘网站抓取多份 **AI Engineer** 职位描述（Job Description），让模型分析并排出最关键的技术技能。

- **输入**：若干职位页 URL（写在 `question` 里）
- **处理**：正则抽 URL → `fetch_website_contents` 抓正文 → 拼进 user prompt
- **输出**：按重要程度排序的技能列表 + 对 Top 3 的一句话理由

## 和本课概念的对照

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| System / User messages | `system_prompt` 定角色与规则，`user_prompt` 塞 JD 正文 |
| 网页抓取 | `scraper.fetch_website_contents` |
| OpenAI 兼容本地接口 | `base_url=http://localhost:11434/v1` + `llama3.2` |
| 多 URL 聚合 | `extract_urls` + 循环抓取拼 prompt |

## 怎么跑

1. `.env` 里准备 `OPENAI_API_KEY`（本笔记本后面演示走的是本地 Ollama；密钥检查格仍会读它）
2. 同目录有 `scraper.py`；需要能访问 CareerViet 等外网职位页
3. 若跑最后一格：本机 Ollama 已启动并 `ollama pull llama3.2`
4. 从上到下依次运行；可在 `question` 里换成你自己的 JD 链接


In [ ]:
# ========== 导入：工具箱 ==========

# 导入标准库 os：读环境变量里的 API Key
import os
# 导入标准库 re：用正则从文本里抽出 URL
import re
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境，避免写死在代码里
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端：后面用 OpenAI 兼容协议调本地 Ollama
from openai import OpenAI
# 从本地 scraper 导入抓取函数：按 URL 取网页正文
from scraper import fetch_website_contents


In [ ]:
# ========== 常量：模型名集中写一处 ==========

# OpenAI 云端小模型名（本笔记本后续未实际调用，但保留常量便于扩展）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境：加载 .env 并粗检 OPENAI_API_KEY ==========

# override=True：.env 中的值覆盖进程里已有同名环境变量
load_dotenv(override=True)

# 读取常见的 OpenAI 密钥环境变量名
api_key = os.getenv("OPENAI_API_KEY")

# 没有密钥时打印排查提示（英文文案保留，便于对照官方排错笔记本）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    # 找到密钥时的提示（原文拼写 founded 保持不动）
    print("API key founded")


In [ ]:
# ========== 试抓：先对单个 JD URL 取正文并数词数 ==========

# 职位页 URL 列表（先放一条做探测；字符串/URL 保持原文）
ai_engineer_url = [
    "https://careerviet.vn/vi/tim-viec-lam/senior-ai-engineer.35C6BE00.html"
]

# 抓取第一页正文，按空白切成词列表，便于粗看页面体量
words = fetch_website_contents(ai_engineer_url[0]).split()
# 打印词数，快速确认抓取是否成功、内容是否够长
print(len(words))


In [ ]:
# ========== 调试：直接打印切词后的正文列表 ==========

# 再次抓同一 URL 并 split，把词列表打印出来（输出可能很长，用于肉眼检查）
print(fetch_website_contents(ai_engineer_url[0]).split())


In [ ]:
# ========== 提示词：系统角色 + 用户侧「问题」（含多条 JD 链接） ==========

# system_prompt：设定「技术招聘分析师」角色，要求排序技能并解释 Top 3
# （整段 prompt 保持英文，直接影响模型输出格式）
system_prompt = """
Role: You are an expert Technical Recruiter and Job Analyst.

Task: Analyze the provided Job Description (JD) and extract all required skills (technical, soft skills, and tools). Focus on the required skill not company's benefits.

Requirements: > 1. Rank the skills in a numbered list from 1 (Most Critical) to N (Least Critical).
2. Ranking Logic: Base the rank on the frequency of mention, the "Required" vs. "Preferred" sections, and how central the skill is to the core responsibilities described.
3. Provide a brief (one-sentence) justification for why the top 3 skills were ranked highest.
"""


# question：用户侧文本；这里主要是一组 AI Engineering 职位链接
# 后面会从中正则抽 URL，再抓各页 JD 拼进真正的 user_prompt
question = f"""
Here are the list of website's links about AI Engineering:
https://careerviet.vn/vi/tim-viec-lam/senior-ai-engineer.35C6BE00.html, https://careerviet.vn/vi/tim-viec-lam/senior-ai-engineer.35C68A6C.html,
https://careerviet.vn/vi/tim-viec-lam/ai-engineer-agentic-llm-systems.35C6CDE2.html, https://careerviet.vn/vi/tim-viec-lam/ai-engineer-computer-vision-npl-llm-khoi-cong-nghe-thong-tin-holt-13.35C6B8DD.html,
https://careerviet.vn/vi/tim-viec-lam/ai-engineer.35C6867E.html
"""


In [ ]:
# ========== 工具函数：从任意文本里抽出 http(s) URL ==========

def extract_urls(user_input):
    # 正则：匹配 http:// 或 https:// 开头，直到空白或逗号为止
    url_pattern = r'https?://[^\s,]+'
    # findall 返回所有匹配到的 URL 字符串列表
    urls = re.findall(url_pattern, user_input)

    # 去掉末尾可能粘上的句号/逗号，得到干净 URL 列表
    return [url.rstrip('.,') for url in urls]


In [ ]:
# ========== 冒烟测试：对 question 跑一遍 URL 抽取 ==========

# 应返回 question 里那几条 CareerViet 链接
extract_urls(question)


In [ ]:
# ========== 组装 user_prompt：原问题文本 + 逐条 JD 正文 ==========

def get_user_prompt(question):
    # 先从 question 文本抽出所有 URL
    url_list = extract_urls(question)
    # 以原始 question 开头，再追加说明（英文拼写 descrtion 保持原样）
    user_prompt = question + "Below are the job descrtion for each job: \n"
    # 枚举每个链接：抓正文并编号拼进 prompt
    for count, link in enumerate(url_list):
        user_prompt += f"Job Description {count+1}: " + fetch_website_contents(link) + "\n"
    # 返回给 Chat Completions 当 user 消息用的长字符串
    return user_prompt


In [ ]:
# ========== 执行：根据 question 生成完整 user_prompt ==========

# 会发起多次网页请求，可能较慢
user_prompt = get_user_prompt(question)


In [ ]:
# ========== 调试：打印拼好的 user_prompt（可能非常长） ==========

print(user_prompt)


In [ ]:
# ========== 组装 messages：system 定规则，user 塞 JD 全文 ==========

# 注意：本格只构建 messages；真正调用模型在后面的 Ollama 格
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]


In [ ]:
# ========== 调试：查看 messages 结构 ==========

print(messages)


In [ ]:
# ========== 路径：用本地 Llama 3.2（Ollama OpenAI 兼容 /v1）回答 ==========

# Ollama 的 OpenAI 兼容基址（需本机 11434 端口在监听）
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key 对本地 Ollama 通常任意非空即可；这里用占位字符串 "ollama"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# 非流式 chat.completions：model 必须与本机已 pull 的名字一致
response = ollama.chat.completions.create(
    model = "llama3.2",
    messages = messages
)
# 取出第一条回复文本并打印
print(response.choices[0].message.content)
